# 0 - Package

In [ ]:
import numpy as np
import pandas as pd
from dataclasses import dataclass
from google.colab import files
from typing import Optional, Tuple, Literal

In [ ]:
# Helper

def export_df_to_excel(
    df: pd.DataFrame,
    file_path: str,
    sheet_name: str = "data",
    index: bool = True,
):
    df.to_excel(
        file_path,
        sheet_name=sheet_name,
        index=index
    )


# 1 - Importer Indice historique


In [ ]:
def load_excel_file():
    """
    Importe un fichier Excel depuis Colab et retourne :
    - le DataFrame complet
    - la colonne Date convertie en datetime
    - la colonne Index convertie en numérique
    """

    uploaded = files.upload()
    filename = next(iter(uploaded))

    df = pd.read_excel(filename)

    dates = pd.to_datetime(df["Date"])

    index = pd.to_numeric(
        df["Index"].astype(str).str.replace(",", "."),
        errors="coerce"
    )

    return df, dates, index

In [ ]:
df1, dates1, index1 = load_excel_file()
print(df1.head())
#print(df2.head())

Saving Data.xlsx to Data.xlsx
        Date        Index
0 2025-12-31  1000.000000
1 2026-01-02   999.726357
2 2026-01-05  1006.618624
3 2026-01-06  1013.130545
4 2026-01-07  1014.727224


## 1.2 - Importer Courbe Taux

In [ ]:
df2, dates2, index2 = load_excel_file()

NameError: name 'load_excel_file' is not defined

In [ ]:
# taux_df = pd.read_excel("Data.xlsx",sheet_name="Taux")

# 2 - Traitement de Time Serie 数据TS清洗


In [ ]:
def load_index_series_df(df: pd.DataFrame) -> pd.Series:
    """
    精简版：加载并清洗包含日期和指数点位的 DataFrame，返回带 DatetimeIndex 的 Series。
    """
    date_col, level_col = df.columns[:2]

    # 使用链式调用一步完成转换、去重与索引设置
    s = (
        df.assign(
            date=pd.to_datetime(df[date_col], errors="coerce"),
            val=pd.to_numeric(
                df[level_col].astype(str).str.replace(" ", "", regex=False).str.replace(",", ".", regex=False),
                errors="coerce"
            )
        )
        .dropna(subset=['date', 'val'])
        .drop_duplicates(subset='date')
        .set_index('date')['val']
        .sort_index()
    )

    if s.empty:
        raise ValueError("Series vide après nettoyage.")
    if (s <= 0).any():
        raise ValueError("Certaines valeurs sont <= 0.")

    return s

In [ ]:
levels = load_index_series_df(df1)
print(levels)
print("--- 检查 DataFrame 的数据类型 ---")
print(levels.dtypes)

date
2025-12-31    1000.000000
2026-01-02     999.726357
2026-01-05    1006.618624
2026-01-06    1013.130545
2026-01-07    1014.727224
                 ...     
2026-09-09    1129.565067
2026-09-10    1122.181018
2026-09-11    1132.994206
2026-09-14    1126.685366
2026-09-15    1117.872007
Name: val, Length: 177, dtype: float64
--- 检查 DataFrame 的数据类型 ---
float64


#3 - Black&Schole Model

In [ ]:
@dataclass
class BSParams:
    r_annual_cc: float
    sigma_annual: float
    s0: float
    s0_date: Optional[pd.Timestamp]

##  3.1 BS Vol imp estimation

In [ ]:
def estimate_sigma_from_history(levels: pd.Series, day_count: int = 365) -> float:
    """
    计算资产的历史年化波动率（Sigma）

    参数:
    - levels: pd.Series，以日期为索引的价格序列
    - day_count: int，年化天数（欧洲市场/日历日通常用 365，股票交易日通常用 252）

    返回:
    - float，年化波动率
    """
    # 1. 严格按时间排序并计算对数收益率
    # 2. 用 replace 把可能产生的无限值(inf)转为 NaN，然后一次性 drop 掉，极具防错性
    log_rets = (
        np.log(levels.sort_index() / levels.shift(1))
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    # 计算样本标准差（ddof=1）并乘以年化系数
    return float(log_rets.std(ddof=1) * np.sqrt(day_count))

## 3.2 (BS)核心参数准备

In [ ]:
def get_s0(
    levels: pd.Series,
    start_date: str,
    default_s0: float = 1000.0,
    use_real: bool = False,
) -> Tuple[float, Optional[pd.Timestamp]]:
    """获取期初价格 S0：支持默认标准化基数或按指定日期提取真实历史价。"""
    if use_real:
        d = pd.Timestamp(start_date)
        if d in levels.index:
            return float(levels.loc[d]), d

    return float(default_s0), None


def build_bs_params_simple(
    levels: pd.Series,
    start_date: str,
    r_neutre_annual_cc: float,
    day_count: int = 365,
    s0_default: float = 1000.0,
    use_real_s0: bool = False,
) -> BSParams:
    """
    一键构建 BS 模拟参数：
      - 自动计算历史波动率 (Sigma)
      - 自动获取或指定 S0
      - 组装并返回 BSParams 对象
    """
    s0, s0_date = get_s0(levels, start_date, s0_default, use_real_s0)
    sigma = estimate_sigma_from_history(levels, day_count=day_count)

    return BSParams(
        r_annual_cc=float(r_neutre_annual_cc),
        sigma_annual=float(sigma),
        s0=s0,
        s0_date=s0_date,
    )

In [ ]:
def get_s0(
    levels: pd.Series,
    start_date: str,
    default_s0: float = 1000.0,
    use_real_if_available: bool = False,
) -> tuple[float, Optional[pd.Timestamp]]:
    """Chọn S0: default hoặc lấy đúng level tại start_date nếu có."""
    if not use_real_if_available:
        return float(default_s0), None

    d = pd.Timestamp(start_date)
    if d in levels.index:
        return float(levels.loc[d]), d
    return float(default_s0), None


def build_bs_params_simple(
    levels: pd.Series,
    start_date: str,
    r_neutre_annual_cc: float,
    day_count: int = 365,
    s0_default: float = 1000.0,
    use_real_s0: bool = False,
) -> BSParams:
    """
    Build params đơn giản:
      - sigma: từ dữ liệu quá khứ
      - r_cc: = r_neutre (bạn truyền vào)
      - s0: default hoặc lấy level thật tại start_date nếu có
    """
    s0, s0_date = get_s0(
        levels,
        start_date=start_date,
        default_s0=s0_default,
        use_real_if_available=use_real_s0,
    )

    sigma = estimate_sigma_from_history(levels, day_count=day_count)

    return BSParams(
        r_annual_cc=float(r_neutre_annual_cc),
        sigma_annual=float(sigma),
        s0=float(s0),
        s0_date=s0_date,
    )

In [ ]:
r_neutre_cc = 0.0691009521484375

params = build_bs_params_simple(
    levels=levels,
    start_date="2025-12-31",
    r_neutre_annual_cc=r_neutre_cc,
    s0_default=1000,
    use_real_s0=False,
)

print(params)

BSParams(r_annual_cc=0.0691009521484375, sigma_annual=0.23032150063543427, s0=1000.0, s0_date=None)


# 4 - Path GBM




## 4.1 - Theory

#### 1. 核心数学公式回顾

几何布朗运动（GBM）的离散化对数收益率公式为：


$$\ln\left(\frac{S_{t+\Delta t}}{S_t}\right) = \left(r - \frac{1}{2}\sigma^2\right)\Delta t + \sigma \sqrt{\Delta t} Z$$


其中 $Z \sim \mathcal{N}(0, 1)$ 是标准正态分布随机数。

你脑海中浮现的“标准 BS 模型公式”通常指的是它的**微分形式（SDE）**：

$$dS_t = r S_t dt + \sigma S_t dW_t$$

而我在前面代码中写的那行公式：


$$\ln\left(\frac{S_{t+\Delta t}}{S_t}\right) = \left(r - \frac{1}{2}\sigma^2\right)\Delta t + \sigma \sqrt{\Delta t} Z$$

并不是凭空捏造的，它是**从标准的微分形式出发，通过伊藤引理（Itô's Lemma）严格推导出的“离散时间下的精确解”**。

在写蒙特卡洛模拟代码时，我们不能直接用微分的 $dt$（计算机无法处理无穷小），必须离散化。如果用最粗糙的欧拉格式去离散化 $dS_t$，会产生较大的截断误差。因此，金融工程中通常会利用下面这个**对数形式的精确转移方程**来进行模拟。

下面我为你**详细推导证明**这个公式是如何从标准 $dS_t$ 变过来的。

---

### 第一步：写出标准的 Black-Scholes 微分方程

资产价格 $S_t$ 服从几何布朗运动（GBM）：


$$dS_t = r S_t dt + \sigma S_t dW_t$$


其中：

* $r$ 是无风险利率（漂移率）。
* $\sigma$ 是波动率。
* $dW_t \sim \mathcal{N}(0, dt)$ 是维纳过程（布朗运动）的增量。

---

### 第二步：运用伊藤引理（Itô's Lemma）

我们想要研究资产价格的对数变化，因此定义一个新变量：


$$X_t = f(S_t) = \ln(S_t)$$

根据**伊藤引理**，对于函数 $f(S_t)$，它的微分 $df$ 可以展开为泰勒展开式的随机版本：


$$df = \frac{\partial f}{\partial t} dt + \frac{\partial f}{\partial S} dS_t + \frac{1}{2} \frac{\partial^2 f}{\partial S^2} (dS_t)^2$$

我们分别求出各个偏导数：

1. $\frac{\partial f}{\partial t} = 0$ （因为公式里没有显式包含时间 $t$）
2. $\frac{\partial f}{\partial S} = \frac{1}{S_t}$
3. $\frac{\partial^2 f}{\partial S^2} = -\frac{1}{S_t^2}$

把这三个偏导数代入伊藤引理公式中：


$$d(\ln S_t) = \frac{1}{S_t} dS_t - \frac{1}{2 S_t^2} (dS_t)^2$$

---

### 第三步：代入 $dS_t$ 并化简

把标准的 $dS_t = r S_t dt + \sigma S_t dW_t$ 代入上式：

1. 第一项：

$$\frac{1}{S_t} dS_t = \frac{1}{S_t} (r S_t dt + \sigma S_t dW_t) = r dt + \sigma dW_t$$


2. 第二项，我们需要先计算 $(dS_t)^2$。在随机积分的运算法则（Itô multiplication table）中：
* $(dt)^2 = 0$
* $dt \cdot dW_t = 0$
* $(dW_t)^2 = dt$


因此：

$$(dS_t)^2 = (r S_t dt + \sigma S_t dW_t)^2 = r^2 S_t^2 (dt)^2 + 2r\sigma S_t^2 dt dW_t + \sigma^2 S_t^2 (dW_t)^2$$



当取极限时，只有最后一项 $(dW_t)^2 = dt$ 保留下来：

$$(dS_t)^2 = \sigma^2 S_t^2 dt$$


3. 把 $(dS_t)^2$ 代回第二项：

$$-\frac{1}{2 S_t^2} (dS_t)^2 = -\frac{1}{2 S_t^2} (\sigma^2 S_t^2 dt) = -\frac{1}{2} \sigma^2 dt$$



---

### 第四步：合并同类项，得到对数微分方程

把上面各部分加起来：


$$d(\ln S_t) = \left( r dt + \sigma dW_t \right) - \frac{1}{2} \sigma^2 dt$$

重新整理，把 $dt$ 提取公因式：


$$d(\ln S_t) = \left( r - \frac{1}{2} \sigma^2 \right) dt + \sigma dW_t$$

这就是**对数价格的随机微分方程**。你会发现，它变成了一个确定性的漂移项 $\left( r - \frac{1}{2} \sigma^2 \right) dt$ 加上一个常数扩散项 $\sigma dW_t$。

---

### 第五步：从微分到积分（离散化到时间步 $\Delta t$）

现在我们把这个微分方程从当前时间 $t$ 积分到下一个观察时间 $t + \Delta t$：


$$\int_t^{t+\Delta t} d(\ln S_u) = \int_t^{t+\Delta t} \left( r - \frac{1}{2} \sigma^2 \right) du + \int_t^{t+\Delta t} \sigma dW_u$$

左边积分结果为：


$$\ln(S_{t+\Delta t}) - \ln(S_t) = \ln\left(\frac{S_{t+\Delta t}}{S_t}\right)$$

右边由于参数是常数，积分非常简单：

* 第一项积分：$\left( r - \frac{1}{2} \sigma^2 \right) \Delta t$
* 第二项积分：布朗运动的增量 $W_{t+\Delta t} - W_t$ 服从正态分布 $\mathcal{N}(0, \Delta t)$。根据正态分布的性质，它可以写成 $\sqrt{\Delta t} Z$，其中 $Z \sim \mathcal{N}(0, 1)$。

最终，我们就拿到了代码里写的那行公式：


$$\ln\left(\frac{S_{t+\Delta t}}{S_t}\right) = \left( r - \frac{1}{2} \sigma^2 \right) \Delta t + \sigma \sqrt{\Delta t} Z$$

---

### 💡 为什么代码里要用这个公式？

1. **数学上是“精确解”**：它没有做任何近似，是通过伊藤引理严格推导出来的。
2. **避免路径爆炸**：如果在代码里直接对 $S_t$ 用欧拉格式（$S_{t+\Delta t} = S_t + r S_t \Delta t + \dots$），当步长不够小时，指数级增长或随机波动很容易让价格算术溢出或失真。而转到对数空间后，变成了**加法运算**（`np.cumsum`），计算机算起来极其稳定且高效。

## 4.2 - 解析code

#### 2. 代码逐行拆解

* **初始化随机数生成器**：
```python
rng = np.random.default_rng(seed)

```


* 使用 NumPy 推荐的现代 API `default_rng`（比旧版的 `np.random.seed` 更安全、性能更好），并支持固定 `seed` 以确保结果可复现。


* **定义时间步长与日期网格**：
```python
dt = 1.0 / 12.0
dates = pd.date_range(start=pd.Timestamp(start_date), periods=n_months + 1, freq="M")

```


* `dt = 1/12` 代表按月模拟（一年 12 个月）。
* `pd.date_range` 生成从起点开始、包含 `n_months + 1` 个月的月末日期序列（包含 $t=0$ 的初始日）。


* **生成标准正态随机数矩阵**：
```python
Z = rng.standard_normal(size=(n_months, n_sims))

```


* 生成一个形状为 `(n_months, n_sims)` 的二维矩阵。**行代表时间步（第 1 个月到第 $N$ 个月），列代表不同的模拟路径（情景）**。


* **计算漂移项与扩散项**：
```python
drift = (r_annual_cc - 0.5 * sigma_annual**2) * dt
diffusion = sigma_annual * np.sqrt(dt) * Z

```


* `drift` 是确定性的漂移率（广播机制会让它自动适配整个矩阵）。
* `diffusion` 是随机震荡项，每一期乘上 $\sqrt{dt}$ 和随机数 $Z$。


* **利用对数的可加性计算累积路径 (`np.cumsum`)**：
这两行代码是整个蒙特卡洛模拟中最巧妙、最精髓的部分！

直接回答你的问题：**`paths` 对应的是 $S_t$（资产在各个未来时点的绝对价格水平）**，**绝对不是** $dS_t$。$dS_t$ 只是价格的微小变化量，而 `paths` 算出来的是我们最终要用的**真实资产价格矩阵**。

为了让你彻底看懂，我们把这两行代码拆开，结合刚才推导的对数公式，一步步看它在数学上到底做了什么：

---

### 第一行代码拆解：`log_paths`

```python
log_paths = np.vstack([np.zeros((1, n_sims)), np.cumsum(drift + diffusion, axis=0)])

```

1. **`drift + diffusion` 是什么？**
这对应着我们刚才推导的单步对数收益率增量（即经过一个时间步长 $\Delta t$ 后的变化）：

$$\Delta \ln(S) = \left( r - \frac{1}{2}\sigma^2 \right)\Delta t + \sigma \sqrt{\Delta t} Z$$



它代表的是“单个月份”内价格对数的变化量。
2. **`np.cumsum(..., axis=0)` 的数学含义（核心！）：**
* **`cumsum`（Cumulative Sum）** 是**累积求和**的意思。
* 为什么要累积？因为对数（$\ln$）有一个极强的数学性质——**可加性**。从今天（$t=0$）到第 3 个月的总对数收益率，等于第 1 个月、第 2 个月、第 3 个月单步对数收益率的**相加**：

$$\ln\left(\frac{S_{t_3}}{S_0}\right) = \ln\left(\frac{S_{t_1}}{S_0}\right) + \ln\left(\frac{S_{t_2}}{S_{t_1}}\right) + \ln\left(\frac{S_{t_3}}{S_{t_2}}\right)$$


* 因此，`np.cumsum` 沿时间轴（`axis=0`）一加，算出来的 `log_paths` 实际上存的是：**从期初 $S_0$ 到未来每个月 $S_t$ 的“总对数收益率”**，即：

$$\text{log\_path}_t = \ln\left(\frac{S_t}{S_0}\right)$$


* 最前面的 `np.zeros((1, n_sims))` 是为了补上 $t=0$ 时的初始状态（因为刚开始时 $\ln(S_0 / S_0) = 0$）。



---

### 第二行代码拆解：`paths`

```python
paths = s0 * np.exp(log_paths)

```

既然我们在上一步通过累加算出了对数比值 $\ln\left(\frac{S_t}{S_0}\right)$，那怎么把它还原成真正的价格 $S_t$ 呢？

1. **取指数（`np.exp`）**：
数学上，指数函数 $\exp$ 是对数函数 $\ln$ 的逆运算。如果对两边同时取指数：

$$\exp\left[ \ln\left(\frac{S_t}{S_0}\right) \right] = \frac{S_t}{S_0}$$



这样就把讨厌的对数消掉了，得到了价格相对于初始价的**倍数**（涨跌幅比例）。
2. **乘以期初价格 `s0` ($S_0$)**：

$$\text{paths} = S_0 \times \frac{S_t}{S_0} = S_t$$



这就完美还原出了**未来每个时间点上，资产的绝对价格 $S_t$**！

---

### 💡 总结成一幅图

这段代码在计算机里构建的矩阵逻辑是这样的：

* **第 0 行 ($t=0$)**：全都是 $S_0$（比如 1000, 1000, 1000...）
* **第 1 个月 ($t=1$)**：通过 $\ln$ 的累加和指数还原，算出各条路径在第 1 个月的真实价格（比如 1020, 985, 1011...）
* **第 2 个月 ($t=2$)**：算出第 2 个月的真实价格...
* ……一直算到最后一个月。

所以，`paths` 就是最终生成的“多条资产价格模拟路径表”，每一行代表一个情景（Scenario），每一列代表一个未来的月份，表格里的每一个数字就是对应的 $S_t$！


* **转置并组装成 DataFrame**：
```python
df = pd.DataFrame(paths.T, index=np.arange(1, n_sims + 1), columns=dates)
df.index.name = "scenario"
return df

```


* `paths.T` 把矩阵转置，变成 **行是情景（`n_sims`），列是日期（`n_months + 1`）**，符合金融回测中“每一行代表一条完整模拟路径”的习惯。





## 4.3 - def

In [ ]:
def simulate_gbm_monthly(
    s0: float,
    r_annual_cc: float,
    sigma_annual: float,
    start_date: str,
    n_months: int,
    n_sims: int,
    seed: Optional[int] = 42,
) -> pd.DataFrame:
    """
    模块 4：基于几何布朗运动 (GBM) 的月度资产路径模拟
    采用 NumPy 内存预分配与对数累加法，性能极致优化。
    返回:
        pd.DataFrame (行 = 情景 scenario，列 = 月末日期 dates)
    """
    if s0 <= 0 or n_months <= 0 or n_sims <= 0:
        raise ValueError("参数 s0、n_months、n_sims 必须大于 0。")

    rng = np.random.default_rng(seed)
    dt = 1.0 / 12.0

    # 内存预分配：(n_months + 1, n_sims)
    log_paths = np.zeros((n_months + 1, n_sims))

    drift = (r_annual_cc - 0.5 * sigma_annual ** 2) * dt
    diffusion_coeff = sigma_annual * np.sqrt(dt)

    # 生成随机数并利用对数可加性进行累加
    z = rng.standard_normal(size=(n_months, n_sims))
    log_paths[1:] = np.cumsum(drift + diffusion_coeff * z, axis=0)

    # 指数还原为绝对价格
    paths = s0 * np.exp(log_paths)
    dates = pd.date_range(start=pd.Timestamp(start_date), periods=n_months + 1, freq="ME")

    return pd.DataFrame(
        paths.T,
        index=pd.Index(range(1, n_sims + 1), name="scenario"),
        columns=dates
    )

In [ ]:
    paths = simulate_gbm_monthly(
        s0=params.s0,
        r_annual_cc=params.r_annual_cc,
        sigma_annual=params.sigma_annual,
        start_date="2025-12-31",
        n_months=145,
        n_sims=10_000,
        seed=42,
    )
    print(paths)

          2025-12-31   2026-01-31   2026-02-28   2026-03-31   2026-04-30  \
scenario                                                                   
1             1000.0  1024.093813  1039.850070   934.609975   876.932036   
2             1000.0   936.506817   997.733179  1062.403013  1062.824567   
3             1000.0  1054.898070   959.928731   959.630856   880.578854   
4             1000.0  1068.316913  1051.968931  1014.427941   970.746690   
5             1000.0   881.462760   934.731810   922.540572   910.909562   
...              ...          ...          ...          ...          ...   
9996          1000.0  1117.691998  1261.313979  1284.614594  1204.849517   
9997          1000.0   999.347586   950.182912   965.698522   901.018011   
9998          1000.0  1009.258707  1056.967334  1201.003574  1334.034050   
9999          1000.0  1081.479675  1124.328844  1157.295685  1093.190521   
10000         1000.0   991.427693  1075.796217  1025.002359  1002.610576   

           

# 5 - decrement

## 5.1 - 理论
这段代码的核心功能是：**对模拟出来的资产价格路径施加“扣减（Decrement）”处理**。

在欧洲的结构化产品市场中，很多标的指数不是普通的“价格指数”或“全收益指数”，而是 **Decrement Index（扣减指数）**。例如，指数会预设每年固定扣除一定点数（比如每年扣 50 点）或固定百分比，用来模拟股息分红或发行商的对冲成本。

下面为你**详细解读其数学逻辑和代码实现**，并提供一个**高度优化、更干净的版本**。

---

### 一、 详细解读与数学逻辑

#### 1. 核心业务逻辑

* **为什么 $t_0$（初始列）不扣减？**
因为期初价格 $S_0$ 是基准点（比如 1000 点），扣减是从期初之后的每一个未来观察期（$t_1, t_2, \dots$）开始累计生效的。
* **两个模式（`mode`）的区别：**
* **`cum`（累积扣减，最常用）**：扣减量随着时间线性累加。如果每年扣 50 点、按月（12期）扣，那么第 1 个月扣 $\frac{50}{12}$ 点，第 2 个月扣 $2 \times \frac{50}{12}$ 点，第 $k$ 个月扣 $k \times \frac{50}{12}$ 点。
* **`flat`（平坦扣减）**：除了 $t_0$ 为 0 外，其余所有后续月份都扣减固定的一期量 $\frac{50}{12}$（不过在实际结构化产品中，绝大多数使用的是 `cum` 累积模式）。



#### 2. 代码逐行拆解

* **日期排序与对齐**：
```python
cols = pd.to_datetime(df_dec.columns)
order = np.argsort(cols.values)
df_dec = df_dec.iloc[:, order]

```


* 确保 DataFrame 的列名（代表日期）是严格按时间先后顺序排列的，防止时间错乱导致扣减顺序颠倒。


* **计算单期扣减基数**：
```python
per_period = float(decrement_value) / float(periods_per_year)  # 例如 50 / 12 = 4.1667
k = np.arange(n_cols, dtype=float)                             # 生成时间步索引 [0, 1, 2, ..., n_cols-1]

```


* **生成扣减序列（一维数组）**：
* 如果是 `flat`：`per_period * (k > 0)` 得到 `[0, 4.1667, 4.1667, 4.1667, ...]`
* 如果是 `cum`：`per_period * k` 得到 `[0, 4.1667, 8.3333, 12.5, ...]`


* **利用 NumPy 广播机制（Broadcasting）一次性减去**：
```python
df_dec.iloc[:, :] = df_dec.values - decrements

```


* `df_dec.values` 是一个形状为 `(n_sims, n_cols)` 的二维矩阵（行是情景，列是日期）。
* `decrements` 是一个形状为 `(n_cols,)` 的一维数组。
* Pandas/NumPy 会自动把这个一维数组“广播”到每一行情景中，实现**所有情景同时减去对应的扣减值**，完全不需要写 `for` 循环，速度极快！

## 5.2 - def

In [ ]:
DecrementMode = Literal["flat", "cum"]

def decrement_paths(
    df: pd.DataFrame,
    decrement_value: float = 50.0,
    periods_per_year: int = 12,
    mode: DecrementMode = "cum",
) -> pd.DataFrame:
    """
    对模拟价格路径施加线性扣减（Decrement Index 模拟）。

    参数:
    - df: 形状为 (n_sims, n_dates) 的模拟路径 DataFrame
    - decrement_value: 每年总扣减量（如点数 50.0）
    - periods_per_year: 每年期数（如月频为 12）
    - mode: "cum"（累积扣减）或 "flat"（固定单期扣减）
    """
    # 1. 确保列名按时间先后排序
    sorted_cols = sorted(pd.to_datetime(df.columns))
    df_dec = df[sorted_cols].copy()

    n_cols = df_dec.shape[1]
    if n_cols == 0:
        return df_dec

    # 2. 计算单期扣减步长并生成时间步数组 k
    per_period = decrement_value / periods_per_year
    k = np.arange(n_cols, dtype=float)

    # 3. 根据模式生成对应的扣减序列
    if mode == "flat":
        decrements = per_period * (k > 0)
    elif mode == "cum":
        decrements = per_period * k
    else:
        raise ValueError("mode 必须是 'flat' 或 'cum'")

    # 4. 利用 NumPy 广播机制直接批量扣减（避免 for 循环）
    df_dec.loc[:, :] = df_dec.values - decrements

    return df_dec

In [ ]:
path_decrement = decrement_paths(paths, decrement_value=50, mode = 'cum')

In [ ]:
path_decrement.head()

,2025-12-31,2026-01-31,2026-02-28,2026-03-31,2026-04-30,2026-05-31,2026-06-30,2026-07-31,2026-08-31,2026-09-30,...,2037-04-30,2037-05-31,2037-06-30,2037-07-31,2037-08-31,2037-09-30,2037-10-31,2037-11-30,2037-12-31,2038-01-31
scenario,,,,,,,,,,,,,,,,,,,,,
1,1000.0,1019.927146,1031.516737,922.109975,860.265369,808.111581,850.571084,896.452415,790.229288,825.964741,...,1180.593815,1048.198523,785.672516,775.707757,662.655715,702.811604,762.021864,768.980529,806.733557,856.947004
2,1000.0,932.340151,989.399845,1049.903013,1046.157900,960.173987,845.902699,826.928811,864.171303,903.245702,...,374.554194,221.527741,254.557994,278.914975,413.081283,394.723699,424.610232,368.791479,429.342927,408.266650
3,1000.0,1050.731404,951.595398,947.130856,863.912187,882.636374,867.348079,962.092710,983.140650,1012.182537,...,1227.790901,1204.540183,1339.724933,1212.933383,1209.597043,1108.209954,1168.267381,1206.718967,1226.220618,1366.440599
4,1000.0,1064.150246,1043.635598,1001.927941,954.080023,963.346994,893.592774,832.860439,796.154561,894.014304,...,880.789321,731.101098,605.928314,547.627441,751.313374,759.923305,799.484114,891.846851,752.829724,656.148033
5,1000.0,877.296094,926.398476,910.040572,894.242895,767.629313,932.218738,829.880390,765.003368,743.323209,...,4861.739510,4494.398723,4466.688639,4782.967769,4738.579987,4955.423603,4898.884998,4263.888573,4270.741837,4254.866012


# 6. Autocall Payoff

## 6.1 - 解析思路

我们可以通过**向量化逻辑（Vectorization）与矩阵运算**，把原本的逐月循环大幅精简，让代码变得**极度清晰、高效且易读**。

### 二、 设计思路与详细步骤拆解

新版本核心的改进思想是：**抛弃原代码中逐个日期循环的 `for` 循环，改用“矩阵全景判定（Matrix Operation）”**。

#### 第一步：构建观察日全景比率矩阵

* **思路**：既然我们需要检查每一个观察日是否满足 Autocall 门槛，不如直接把所有**符合频率要求的观察日数据**抽出来，组成一个大矩阵。
* **数学表达**：
设矩阵大小为 $M \times N$（$M$ 条情景，$N$ 个观察日），其中每个元素为：

$$\text{Ratio}_{i, j} = \frac{S_{i, t_j}}{S_{\text{ref}}}$$



#### 第二步：利用布尔矩阵与 `argmax` 一次性找出“最早触发点”

* **思路**：在原代码中，我们是用 `for` 循环一个月一个月往下找的。如果某条情景在第 2 个月就触发了 Autocall，后面的月份其实不需要再管了。
* **矩阵优化**：
1. 我们可以直接对整个比率矩阵进行比较：`hit_matrix = (Ratio >= Barrier)`。这会得到一个由 `True` 和 `False` 组成的判定矩阵。
2. 如果某条情景在某些月份触发了，对应的位置就是 `True`。如何找到**第一次**触发的月份呢？
3. 利用 NumPy 的 `.argmax(axis=1)`，它会沿着时间轴（横向）寻找**第一个出现 `True` 的列索引**。这就相当于在一瞬间找出了每条情景最早被提前赎回的时间点，完全省去了 Python 的 `for` 循环！



#### 第三步：对“未触发（Not Called）”的情景进行期末分类

* **思路**：对于没有被提前赎回的情景，它们的命运决定于最终到期日（Maturity）的表现 $r_T = \frac{S_T}{S_{\text{ref}}}$。
* **分段函数合并**：
我们直接通过 Pandas 的布尔掩码（Mask），把所有情景归纳进对应的收益公式中：
* **高收益/提前赎回组**：`Called` 或者 `rT >= barriere_sortie_maturite` $\rightarrow$ 拿本金加利息：

$$\text{Payoff} = S_{\text{ref}} \times (1 + c \cdot k)$$


* **保本组**：`barriere_protection <= rT < barriere_sortie_maturite` $\rightarrow$ 拿回 100% 本金：

$$\text{Payoff} = S_{\text{ref}}$$


* **亏损组**：`rT < barriere_protection` $\rightarrow$ 承担实际跌幅：

$$\text{Payoff} = S_{\text{ref}} \times r_T$$





#### 第四步：计算存续时间与结果组装

* **思路**：通过退出日期（`exit_date`）减去初始日期，除以 `day_count` 得到年化时间 $T$：

$$T = \frac{\text{Date}_{\text{sortie}} - \text{Date}_{\text{initiale}}}{\text{day\_count}}$$


* 最后将所有核心风控与估值指标（$S_0$、$S_{\text{ref}}$、退出日期、最终比率、期数、Payoff、时间 $T$、是否被 Called）对齐并封装成整洁的报表返回。

## 6.2 - def

In [ ]:
FreqSortie = Literal["mensuelle", "trimestrielle", "annuelle"]

def _freq_to_months(freq_sortie: FreqSortie) -> int:
    """将观察频率转换为对应的月份数"""
    mapping = {"mensuelle": 1, "trimestrielle": 3, "annuelle": 12}
    if freq_sortie not in mapping:
        raise ValueError("freq_sortie 必须是 'mensuelle', 'trimestrielle' 或 'annuelle'。")
    return mapping[freq_sortie]

def Payoff(
    paths: pd.DataFrame,
    s_ref: float,
    barriere_sortie_anticipe: float,
    barriere_sortie_maturite: float,
    barriere_protection: float,
    premiere_annee_sortie: int,
    freq_sortie: FreqSortie,
    coupon_par_periode: float,
    annee_finale: int,
    day_count: int = 365,
) -> pd.DataFrame:
    """
    高级优化版 SAF 产品 Payoff 计算引擎：
    基于矩阵运算（向量化），一次性处理所有情景的 Autocall 触发和期末兑付。
    """
    if paths.shape[1] < 2:
        raise ValueError("paths 至少需要包含 2 列日期。")

    # 1. 规范化日期与基础数据提取
    df = paths.loc[:, sorted(pd.to_datetime(paths.columns))].copy()
    df.columns = pd.to_datetime(df.columns)

    scenarios = df.index
    date_initiale = df.columns[0]
    S_0 = df.iloc[:, 0].astype(float)

    # 2. 筛选合法的观察日矩阵
    step_months = _freq_to_months(freq_sortie)
    start_obs_date = date_initiale + pd.DateOffset(years=premiere_annee_sortie)
    maturity_target = date_initiale + pd.DateOffset(years=annee_finale)

    # 获取期限内的所有有效列，最后一列作为到期日
    valid_cols = df.columns[df.columns <= maturity_target]
    if len(valid_cols) == 0:
        raise ValueError("没有找到符合 maturity_target 的有效日期。")
    maturity_date = valid_cols[-1]

    # 计算各列相对于起始日的月份差
    months_diff = ((valid_cols.year - date_initiale.year) * 12 + (valid_cols.month - date_initiale.month))

    # 筛选出满足“到达首个观察年”且“符合观察频率”的列
    obs_mask = (valid_cols >= start_obs_date) & (months_diff % step_months == 0)
    obs_dates = valid_cols[obs_mask]

    # 3. 核心向量化计算：Autocall 触发判定
    # 计算所有观察日的价格比率矩阵 (shape: n_scenarios x n_obs_dates)
    if len(obs_dates) > 0:
        ratio_obs_matrix = df[obs_dates].astype(float) / float(s_ref)
        hit_matrix = ratio_obs_matrix >= barriere_sortie_anticipe

        # 找到每条情景中“最早”被触发的索引（如果都没触发，返回默认值）
        has_hit = hit_matrix.any(axis=1)
        # argmax 会返回第一个 True 的位置索引
        first_hit_idx = hit_matrix.values.argmax(axis=1)

        # 构造每个情景的退出日期和期数
        exit_date = pd.Series(maturity_date, index=scenarios, dtype="datetime64[ns]")
        nb_periodes = pd.Series(index=scenarios, dtype=int)
        ratio_sortie = pd.Series(index=scenarios, dtype=float)

        # 被提前触发的情景
        if has_hit.any():
            hit_scenarios = scenarios[has_hit]
            hit_positions = first_hit_idx[has_hit]

            exit_date.loc[hit_scenarios] = obs_dates[hit_positions]
            nb_periodes.loc[hit_scenarios] = months_diff[obs_mask][hit_positions] // step_months
            ratio_sortie.loc[hit_scenarios] = ratio_obs_matrix.values[np.where(has_hit)[0], hit_positions]

        called = has_hit
    else:
        called = pd.Series(False, index=scenarios)
        exit_date = pd.Series(maturity_date, index=scenarios, dtype="datetime64[ns]")
        nb_periodes = pd.Series(index=scenarios, dtype=int)
        ratio_sortie = pd.Series(index=scenarios, dtype=float)

    # 4. 处理未被提前触发（Called = False）的情景
    not_called = ~called
    maturity_months = (maturity_date.year - date_initiale.year) * 12 + (maturity_date.month - date_initiale.month)
    maturity_periods = maturity_months // step_months

    nb_periodes.loc[not_called] = maturity_periods

    # 计算到期比率 rT
    sT = df[maturity_date].astype(float)
    rT = (sT / float(s_ref)).astype(float)
    ratio_sortie.loc[not_called] = rT.loc[not_called]

    # 5. 计算最终 Payoff 收益
    payoff = pd.Series(index=scenarios, dtype=float)

    # Called 或 期末达标 (rT >= barriere_sortie_maturite)
    condition_high = called | (not_called & (rT >= barriere_sortie_maturite))
    payoff.loc[condition_high] = float(s_ref) * (1.0 + coupon_par_periode * nb_periodes.loc[condition_high].astype(float))

    # 期末平稳区间 (barriere_protection <= rT < barriere_sortie_maturite)
    condition_flat = not_called & (rT >= barriere_protection) & (rT < barriere_sortie_maturite)
    payoff.loc[condition_flat] = float(s_ref)

    # 期末大跌区间 (rT < barriere_protection)
    condition_low = not_called & (rT < barriere_protection)
    payoff.loc[condition_low] = float(s_ref) * rT.loc[condition_low].astype(float)

    # 6. 计算年化时间 T 并打包报表
    T = (pd.to_datetime(exit_date) - pd.to_datetime(date_initiale)).dt.days.astype(float) / float(day_count)

    return pd.DataFrame({
        "Date_pricing": date_initiale,
        "Date_initiale": date_initiale,
        "S_0": S_0.values,
        "S_ref": float(s_ref),
        "Date_sortie": exit_date.values,
        "Ratio_sortie_S_sur_Sref": ratio_sortie.values,
        "Nb_periodes": nb_periodes.values,
        "Payoff": payoff.values,
        "T_en_annees": T.values,
        "Called": called.values,
    }, index=scenarios)

In [ ]:
pay_tbl = Payoff(
    paths=path_decrement,
    s_ref=1000.0,
    barriere_sortie_anticipe=0.86,
    barriere_sortie_maturite=0.86,
    barriere_protection=0.60,
    premiere_annee_sortie=2,
    freq_sortie="trimestrielle",
    coupon_par_periode=0.019,
    annee_finale=12,
)

# 7. 市场价值VM

## 7.1 - 解析code
这段代码是结构化产品定价引擎的**现值计算（Valuation / Present Value）**核心。它的作用是：结合上一步算出来的每条情景的 Payoff（收益）和存续时间 $T$，再对照**零息利率曲线（Yield Curve）**进行插值和折现，最终求出所有情景折现后的平均值，也就是产品的**公允价值（VM, Valeur de Marché / Market Value）**。

### 二、 详细设计思路与步骤拆解

这段代码虽然看起来不长，但包含了金融工程中非常严谨的**收益率曲线处理、利率插值、复利计算与风险中性定价原理**。我们将其拆为 5 个部分逐一解析：

#### 1. `_parse_rate_to_float`：防错的利率解析器

* **思路**：真实业务中，利率数据的来源五花八门——可能是小数（`0.0259`）、百分数数字（`2.59` 代表 2.59%），也可能是带百分号或欧洲逗号的字符串（`'2,59%'`）。
* **数学与逻辑**：
* 用 `pd.isna` 拦截空值。
* 如果是数字且大于 1（比如 `2.59`），自动判定为百分数，除以 100 转为小数（`0.0259`）。
* 如果是字符串，通过 `.strip().replace("%", "").replace(",", ".")` 剥离百分号、将欧洲逗号换成小数点，再统一归一化为纯小数。



#### 2. 利率曲线（Yield Curve）的标准化处理

```python
    curve = courbe_taux[[col_year, col_taux]].dropna().copy()
    curve[col_year] = curve[col_year].astype(float)
    curve[col_taux] = curve[col_taux].apply(_parse_rate_to_float).astype(float)
    curve = curve.sort_values(col_year)
    x = curve[col_year].to_numpy(dtype=float)
    y = curve[col_taux].to_numpy(dtype=float)

```

* **思路**：输入的利率曲线（`courbe_taux`）通常只有几个关键期限点（比如 1年期、2年期、3年期、5年期利率）。我们需要把它们提取出来，变成干净的横坐标数组 $x$（期限 年）和纵坐标数组 $y$（利率 $r$），并按期限升序排序，为接下来的插值做准备。

#### 3. 线性插值（Linear Interpolation）与边界限制

```python
    T = tbl["T_en_annees"].astype(float).to_numpy()
    T_clip = np.clip(T, x.min(), x.max())
    rT = np.interp(T_clip, x, y)

```

* **为什么需要插值？**
产品的存续时间 $T$（比如 1.45 年、2.18 年）是连续的、五花八门的，而利率曲线只有离散的点（比如 1年和 2年利率）。我们需要用**线性插值**算出任意存续时间 $T$ 对应的准确无风险利率 $r(T)$。
* **`np.clip` 的作用**：防止某些情景的 $T$ 超出了利率曲线的最大/最小期限范围（比如曲线只给到 5 年，但产品由于某种原因算出来 5.1 年），使用 `np.clip` 把 $T$ 强制夹在曲线区间内，避免插值函数崩溃。

#### 4. 计算折现因子（Discount Factor, DF）

根据金融工程中不同的复利惯例（`compounding`），折现公式分为两种：

* **连续复利（Continuous Compounding, `"cc"`）**：

$$\text{DF}(T) = e^{-r(T) \cdot T}$$


* **年度复利（Annual Compounding, `"annual"`）**：

$$\text{DF}(T) = (1 + r(T))^{-T}$$


* **代码实现**：通过 `np.exp` 或 `np.power` 对整个向量进行底层加速计算，瞬间算出每一条情景对应的折现系数。

#### 5. 计算现值（Present Value, PV）与公允价值（VM）

```python
    tbl["Taux_interp"] = rT
    tbl["Facteur_actualisation"] = df_factor
    tbl["PV"] = tbl["Payoff"].astype(float).to_numpy() * df_factor

    vm = float(tbl["PV"].mean())
    return tbl, vm

```

* **每条情景的现值**：

$$\text{PV}_i = \text{Payoff}_i \times \text{DF}(T_i)$$


* **公允价值（VM, Value de Marché）**：
根据风险中性定价原理，公允价值等于所有情景现值的算术平均数（大数定律）：

$$\text{VM} = \frac{1}{N} \sum_{i=1}^{N} \text{PV}_i = \mathbb{E}[\text{PV}]$$


* 最后将包含插值利率、折现因子、每条情景 PV 的详细报表 `tbl` 与总公允价值 `vm` 一并返回，完美收官整个定价流程！

In [ ]:
Compounding = Literal["cc", "annual"]

def _parse_rate_to_float(x) -> float:
    """健壮的利率文本/数值解析器：支持 0.0259, 2.59, '2,59%', '2.59%' 等格式。"""
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.floating)):
        v = float(x)
        return v / 100.0 if v > 1.0 else v

    # 统一清洗字符串格式并转为浮点数
    s = str(x).strip().replace("%", "").replace(",", ".")
    v = float(s)
    return v / 100.0 if v > 1.0 else v

def VM(
    payoff_table: pd.DataFrame,
    courbe_taux: pd.DataFrame,
    col_year: str = "Year",
    col_taux: str = "Taux",
    compounding: Compounding = "annual",
) -> Tuple[pd.DataFrame, float]:
    """
    高级优化版现值与公允价值（VM）计算引擎：
    根据收益表与零息利率曲线，进行线性插值、折现因子计算并返回详细报表及公允价值。
    """
    tbl = payoff_table.copy()

    # 1. 利率曲线清洗与排序
    curve = courbe_taux[[col_year, col_taux]].dropna().copy()
    curve[col_year] = curve[col_year].astype(float)
    curve[col_taux] = curve[col_taux].apply(_parse_rate_to_float).astype(float)
    curve = curve.sort_values(col_year)

    x = curve[col_year].to_numpy(dtype=float)
    y = curve[col_taux].to_numpy(dtype=float)

    # 2. 提取每条情景的存续年限 T
    T = tbl["T_en_annees"].astype(float).to_numpy()

    # 3. 利率插值 (使用 np.interp 并限制边界以防外推报错)
    T_clip = np.clip(T, x.min(), x.max())
    rT = np.interp(T_clip, x, y)

    # 4. 根据复利类型计算折现因子 (Discount Factor)
    if compounding == "cc":
        df_factor = np.exp(-rT * T)
    elif compounding == "annual":
        df_factor = np.power(1.0 + rT, -T)
    else:
        raise ValueError("compounding 必须是 'annual' 或 'cc'。")

    # 5. 整合结果并计算平均公允价值 (VM)
    tbl["Taux_interp"] = rT
    tbl["Facteur_actualisation"] = df_factor
    tbl["PV"] = tbl["Payoff"].astype(float).to_numpy() * df_factor

    vm = float(tbl["PV"].mean())
    return tbl, vm

In [ ]:
# courbe_taux lấy từ sheet taux (Year, Taux)
tbl_vm, vm = VM(
    payoff_table=pay_tbl,
    courbe_taux=taux_df,
    col_year="Year",
    col_taux="Taux",
    compounding="annual"  # hoặc "cc" nếu bạn chắc là taux en continu
)

vm

NameError: name 'taux_df' is not defined